In [ ]:
# imports
%load_ext autoreload
%autoreload all
import sys, os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from src.mnist_mlp import MLP, MLP_NO_OUTPUT_ACTIVATION
from src import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import itertools, json
import plotly.express as px

In [ ]:
# Plot validation loss in reference to number of params and number of hidden layers
# parameters = {
#     "num_hidden_layers": range(1, 10, 2),
#     "hidden_layer_size": range(64, 257, 64),
#     "activation": ["tanh", "relu"],
#     "lr": np.logspace(-4, -1, 20),
#     "model": [MLP, MLP_NO_OUTPUT_ACTIVATION],
# }
parameters = {
    "num_hidden_layers": [1, 3],
    "hidden_layer_size": [128, 256],
    "activation": ["tanh", "relu"],
    "lr": [3e-4, 1e-3],
    "model": [MLP, MLP_NO_OUTPUT_ACTIVATION],
}

epochs = 40
batch_size = 32

In [ ]:
# load MNIST data
train_img_path = "../data/train_img.idx"
train_label_path = "../data/train_label.idx"
test_img_path = "../data/test_img.idx"
test_label_path = "../data/test_label.idx"

train_samples, train_labels, test_samples, test_labels = normalize_mnist_data(
    train_img_path, train_label_path, test_img_path, test_label_path
)

# use tensors
train_samples = torch.from_numpy(train_samples).float()
train_labels = torch.from_numpy(train_labels)
test_samples = torch.from_numpy(test_samples).float()
test_labels = torch.from_numpy(test_labels)

In [ ]:
def benchmark_parameters(
    model: nn.Module,
    train_samples: torch.Tensor,
    train_labels: torch.Tensor,
    test_samples: torch.Tensor,
    test_labels: torch.Tensor,
    num_hidden: int,
    size_hidden: int,
    activation: str,
    epochs: int,
    batch_size: int,
    lr: float,
    optim: str = "adam",
) -> tuple[float, float, int]:
    # setup network
    mlp = model(784, 10, np.array(num_hidden * [size_hidden]), activation)
    if optim == "adam":
        optimizer = torch.optim.AdamW(mlp.parameters(), lr)
    else:
        optimizer = torch.optim.SGD(mlp.parameters(), lr)
    best_value = {
        "epoch": 0,
        "valid_accuracy": 0.0,
        "valid_loss": 0.0,
        "train_accuracy": 0.0,
        "train_loss": 0.0,
    }

    # train model
    for epoch in range(epochs):
        # randomize order of training samples and labels
        idx = torch.randperm(len(train_samples))
        train_samples = train_samples[idx]
        train_labels = train_labels[idx]
        for i in range(0, len(train_samples), batch_size):
            # forward pass
            optimizer.zero_grad()
            y_hat = mlp(train_samples[i : i + batch_size].flatten(1))
            loss = F.cross_entropy(y_hat, train_labels[i : i + batch_size].flatten())
            # backward pass
            loss.backward()
            optimizer.step()

        # test the model
        validation_accuracy = get_accuracy(mlp, test_samples, test_labels)
        validation_loss = F.cross_entropy(
            mlp(test_samples), test_labels.flatten()
        ).item()
        training_accuracy = get_accuracy(mlp, train_samples, train_labels)
        training_loss = F.cross_entropy(
            mlp(train_samples), train_labels.flatten()
        ).item()

        # not quite early stopping -> test for accuracy changes
        if validation_accuracy > best_value["valid_accuracy"]:
            best_value = {
                "epoch": epoch,
                "valid_accuracy": validation_accuracy,
                "valid_loss": validation_loss,
                "train_accuracy": training_accuracy,
                "train_loss": training_loss,
            }
    return (
        validation_accuracy,
        validation_loss,
        training_accuracy,
        training_loss,
        mlp.param_count(),
        best_value,
        mlp._get_name(),
    )

In [ ]:
parameters.values()

In [ ]:
best_accuracy = 0.0
best_params = {}
parameters_used = []

for hidden_num, hidden_size, activation, lr, model in itertools.product(
    parameters["num_hidden_layers"],
    parameters["hidden_layer_size"],
    parameters["activation"],
    parameters["lr"],
    parameters["model"],
):
    (
        validation_accuracy,
        validation_loss,
        training_accuracy,
        training_loss,
        parameter_count,
        best_value,
        model_name,
    ) = benchmark_parameters(
        model,
        train_samples.flatten(1),
        train_labels,
        test_samples.flatten(1),
        test_labels,
        hidden_num,
        hidden_size,
        activation,
        epochs,
        batch_size,
        lr,
        "adam",
    )
    params = {
        "num_hidden_layers": hidden_num,
        "hidden_layer_size": hidden_size,
        "activation": activation,
        "learning_rate": lr,
        "validation_accuracy": validation_accuracy,
        "validation_loss": validation_loss,
        "training_accuracy": training_accuracy,
        "training_loss": training_loss,
        "num_params": parameter_count,
        "best_value": best_value,
        "model": model_name,
    }
    parameters_used.append(params)
    if validation_accuracy > best_accuracy:
        best_accuracy = validation_accuracy
        best_params = params

    print(f"best: {best_params} \ncurrent: {params}")
    with open("../data/parameters_used_test.json", "w") as file:
        json.dump(parameters_used, file, indent=4)

In [ ]:
with open("../data/parameters_used_test.json", "r") as file:
    parameters_used = json.load(file)

# plot the results
num_hidden_layers = [result["num_hidden_layers"] for result in parameters_used]
hidden_layer_size = [result["hidden_layer_size"] for result in parameters_used]
num_params = [result["num_params"] for result in parameters_used]
validation_accuracy = [result["validation_accuracy"] for result in parameters_used]
validation_loss = [result["validation_loss"] for result in parameters_used]
training_accuracy = [result["training_accuracy"] for result in parameters_used]
training_loss = [result["training_loss"] for result in parameters_used]
learning_rate = [result["learning_rate"] for result in parameters_used]
activation = [result["activation"] for result in parameters_used]
model = [result["model"] for result in parameters_used]

df = pd.DataFrame(
    {
        "num_hidden_layers": num_hidden_layers,
        "hidden_layer_size": hidden_layer_size,
        "num_params": num_params,
        "learning_rate": learning_rate,
        "validation_accuracy": validation_accuracy,
        "validation_loss": validation_loss,
        "training_accuracy": training_accuracy,
        "training_loss": training_loss,
        "activation": activation,
        "model": model,
    }
)

df_tanh = df[df["activation"] == "tanh"]

df_relu = df[df["activation"] == "relu"]

In [ ]:
df_lr = df[(df["num_hidden_layers"] == 3)]

fig = px.line(
    df_lr,
    x="learning_rate",
    y=["validation_accuracy", "training_accuracy"],
    color="activation",
    log_x=True,
    facet_row="hidden_layer_size",
    facet_col="model",
    height=1200,
    markers=True,
)
fig.show()

In [ ]:
df_lr = df[(df["num_hidden_layers"] == 1)]

fig = px.line(
    df_lr,
    x="learning_rate",
    y="validation_accuracy",
    color="activation",
    log_x=True,
    facet_row="hidden_layer_size",
    facet_col="model",
    height=1200,
)
fig.show()

In [ ]:
fig = px.line(
    df_relu,
    x="hidden_layer_size",
    y=["validation_accuracy", "training_accuracy"],
    color="num_hidden_layers",
    text="activation",
    facet_row="learning_rate",
    facet_col="model",
)
fig.add_traces(
    px.line(
        df_tanh,
        x="hidden_layer_size",
        y=["validation_accuracy", "training_accuracy"],
        color="num_hidden_layers",
        text="activation",
        facet_row="learning_rate",
        facet_col="model",
    ).data
)
fig.show()

# TODO:
- 
- 